# 07 — Publish Gold governance check

AIDP operational pipeline notebook. It uses Volumes and governed tables only; the Git checkout is code only.

In [ ]:
from pathlib import Path
import os
import sys
from pyspark.sql import SparkSession, functions as F, Window
spark = SparkSession.builder.getOrCreate()
root = Path.cwd().resolve()
for candidate in (Path(os.environ['AIDP_PROJECT_ROOT']).resolve(), root, *root.parents) if os.getenv('AIDP_PROJECT_ROOT') else (root, *root.parents):
    if (candidate / 'src').is_dir() and (candidate / 'config/settings.yaml').is_file():
        root = candidate
        break
else:
    raise RuntimeError('Project code root not found. Clone the repository; do not use it as a data store.')
sys.path.insert(0, str(root / 'src'))
from aidp_pipeline import LANDING_COUNTS, RAW_SESSIONS, STRUCTURAL_KEYS, landing_paths, load_aidp_config, missing_landing_paths, table_name
config = load_aidp_config(root / 'config/settings.yaml')


In [ ]:
historical = ['observed_driver_profiles','historical_pace_summary','historical_variability']
scenario = ['virtual_max_profile','deterministic_chase_summary','monte_carlo_scenario_summary','synthetic_field_summary']
for name in historical:
    table = table_name(config, 'gold_historical_schema', name)
    if not spark.catalog.tableExists(table): raise RuntimeError(f'Missing Gold Historical table: {table}')
    print('Gold Historical:', table)
for name in scenario:
    table = table_name(config, 'gold_scenario_schema', name)
    print('Gold Scenario:', table, 'exists=', spark.catalog.tableExists(table))
print('Governance confirmed: historical evidence and synthetic what-if outputs use separate namespaces.')
